# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to explore and analyze the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) using the `mlcroissant` library and Python tools.

### Dataset Source
The dataset is provided via a [Croissant](https://mlcommons.org/croissant/) schema URL, making it machine-actionable and AI-ready for reproducible data analysis.

In [ ]:
# Install mlcroissant if needed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and explore high-level information about the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant metadata URL:
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset loaded:\n")
print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.date_published}")


## 2. Data Overview
Let's list the available record sets and the fields they contain (referenced by their `@id`).
This helps us understand the organization of the dataset before extracting data.

In [ ]:
# List record sets and print their fields, all by @id
for record_set in dataset.record_sets:
    print(f"RecordSet: {record_set.name} (id: {record_set.id})")
    print("  Fields (by @id):")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print("")

## 3. Data Extraction
Load the main survey response record set, by its `@id`, into a pandas DataFrame for further analysis.
You may repeat this for other record sets if present.

In [ ]:
# Identify the main record set id (by inspection)
# As an example, we will select the first record set. Adjust if multiple/reduce if only one exists.
main_record_set = None
for record_set in dataset.record_sets:
    main_record_set = record_set.id
    break
print(f"Selected main record set: {main_record_set}")

# List all available record sets by ID
record_set_ids = [rs.id for rs in dataset.record_sets]

# Load all record sets into dataframes
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records for record set: {record_set_id}")

# Preview the main dataframe
if main_record_set in dataframes:
    print(f"Columns in main record set: {dataframes[main_record_set].columns.tolist()}")
    dataframes[main_record_set].head()
else:
    print("No records found for the main record set.")

## 4. Exploratory Data Analysis (EDA)
Apply simple EDA: filter for high GAD-7 anxiety scores and normalize the PHQ-9 depression scores.
All columns are referenced by their `@id`, as defined in the Croissant schema.

**Note:** If the real field IDs differ or fields are named differently, inspect the previous cell's printed column names and adjust the variables accordingly below.

In [ ]:
# Update these @id variables as appropriate for your dataset keys
record_set_id = main_record_set

# Example field @ids (update as needed by inspecting columns)
gad7_score_id = None
phq9_score_id = None
education_id = None
for col in dataframes[record_set_id].columns:
    # Search for typical field names that might represent their respective scores
    if 'gad' in col.lower() and 'score' in col.lower():
        gad7_score_id = col
    if 'phq' in col.lower() and 'score' in col.lower():
        phq9_score_id = col
    if 'education' in col.lower():
        education_id = col

print(f"GAD-7 Score field: {gad7_score_id}")
print(f"PHQ-9 Score field: {phq9_score_id}")
print(f"Education field: {education_id}")

# Proceed if fields are found
if gad7_score_id and phq9_score_id:
    df = dataframes[record_set_id]
    # Filter: GAD-7 score above threshold
    threshold = 10
    filtered_df = df[df[gad7_score_id] > threshold].copy()
    print(f"Filtered records with {gad7_score_id} > {threshold} (N={len(filtered_df)}):")
    display_cols = [gad7_score_id]
    if education_id:
        display_cols.append(education_id)
    print(filtered_df[display_cols].head())

    # Normalize PHQ-9 scores
    filtered_df[f"{phq9_score_id}_normalized"] = (
        filtered_df[phq9_score_id] - filtered_df[phq9_score_id].mean()
    ) / filtered_df[phq9_score_id].std()
    print(f"\nNormalized {phq9_score_id} for filtered records:")
    print(filtered_df[[phq9_score_id, f"{phq9_score_id}_normalized"]].head())

    # Group by education (if available)
    if education_id and education_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(education_id)[phq9_score_id].mean().reset_index()
        print(f"\nMean PHQ-9 score grouped by education ({education_id}):")
        print(grouped_df)
else:
    print("Could not find GAD-7 or PHQ-9 score fields. Adjust field IDs as needed.")

## 5. Visualization
Let's visualize the distribution of PHQ-9 normalized scores in the subset with high GAD-7 scores, and (if available) overlay this by education group.

*Tip: If you encounter errors related to missing fields, check the printed columns from above and adjust the variables accordingly.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if normalization field exists
if gad7_score_id and phq9_score_id and f"{phq9_score_id}_normalized" in filtered_df:
    plt.figure(figsize=(8, 5))
    if education_id and education_id in filtered_df.columns:
        sns.histplot(data=filtered_df, x=f"{phq9_score_id}_normalized", hue=education_id, multiple="stack", kde=True)
        plt.title(f"Distribution of Normalized PHQ-9 Scores (> GAD-7 Threshold), by Education")
    else:
        sns.histplot(filtered_df[f"{phq9_score_id}_normalized"], kde=True)
        plt.title("Distribution of Normalized PHQ-9 Scores for High GAD-7 Participants")
    plt.xlabel("PHQ-9 (normalized)")
    plt.tight_layout()
    plt.show()
else:
    print("Visualization skipped: required fields not found.")

## 6. Conclusion
We have successfully loaded, inspected, and performed simple EDA on the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

- **Data loading** is streamlined using Croissant schemas
- **Record sets and fields** are referenced using their persistent `@id`s
- **EDA** reveals how GAD-7 and PHQ-9 scores may co-vary and how demographic factors (e.g. education) can be analyzed

This approach provides a reproducible baseline for more in-depth analysis, and can be extended to other datasets adopting FAIR, schema-based standards.